In [ ]:
"""
Multi-Factor Portfolio Risk & Attribution Engine
Calculates historical Value at Risk (VaR), Expected Shortfall, and Maximum Drawdown
for an equal-weighted equity portfolio. Conducts asset-level factor attribution
using the Fama-French 3-Factor Model (OLS Regression) and historical stress testing.
"""
import yfinance as yf
import pandas as pd
from sklearn.linear_model import LinearRegression
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
"""
1. DATA INGESTION & PORTFOLIO FORMULATION
"""
stocks = ["AAPL", "GOOGL", "MSFT"] #choose any stock you need
# Change your download line to this:
download = yf.download(stocks, start="2020-01-01", end="2026-01-01", auto_adjust=False)
prices = download["Close"]
returns = prices.pct_change().dropna()
#formulate equal-weighted portfolio allocation
weights = [1/3,1/3,1/3]
portfolio_returns = (returns * weights).sum(axis=1)
portfolio_variance = portfolio_returns.var()
#load and clean Fama-French 3-Factor daily data
factors = pd.read_csv("/F-F_Research_Data_Factors_daily.csv", skiprows=3)
factors.rename(columns={factors.columns[0]: "Date"}, inplace=True)
factors["Date"] = factors["Date"].astype(str).str.strip()
factors = factors[factors["Date"].str.len()==8].copy()
factors["Date"] = pd.to_datetime(factors["Date"], format="%Y%m%d") #changes the date format to ours
factors.set_index("Date", inplace=True)
factors = factors.astype(float)/100
#align indices and calculate excess portfolio returns
df = pd.concat([portfolio_returns.rename("Portfolio"), factors], axis=1).dropna()
df["Port Excess"] = df['Portfolio'] - df["RF"]
print(f"Daily portfolio variance is: {portfolio_variance:.6f}")
"""
2. MULTI-FACTOR OLS REGRESSION
"""
x = df[["Mkt-RF","SMB","HML"]]
y = df["Port Excess"]
reg = LinearRegression().fit(x,y)
print(f"Alpha: {reg.intercept_:.6f}") #unexplained excess return
for factor, beta in zip(x.columns, reg.coef_):
  print(f"Beta {factor}: {beta:.4f}") #sensitivity to market forces
print(f"R-score: {reg.score(x,y):.4f}")
"""
3. DOWNSIDE RISK METRICS (VaR & CVaR)
"""
var_95 = np.percentile(df["Portfolio"],5) #historical Value at Risk (VaR) at 95% confidence interval
cvar_95 = df["Portfolio"][df["Portfolio"] <= var_95].mean() #expected shortfall
print(f"Value at Risk(VaR):{var_95:.4f}")
print(f"Expected shortfall: {cvar_95:.4f}")
#
wealth_index = (1+df["Portfolio"]).cumprod()
previous_peaks = wealth_index.cummax()
drawdowns = (wealth_index - previous_peaks)/previous_peaks
max_drawdown = drawdowns.min()
print(f"Maximal drawdown: {max_drawdown:.2f}")
"""
4. HISTIRICAL STRESS TESTING
"""
events = {"Covid-19": ("2020-02-19", "2020-03-23"),"2022 Tech Selloff": ("2022-01-03", "2022-10-12")}
for event, (start,end) in events.items():
  start_dt = pd.to_datetime(start)
  end_dt = pd.to_datetime(end)
  period_df = df.loc[start_dt:end_dt]
  if not period_df.empty:
    portfolio_loss = (1+ period_df["Portfolio"]).prod()-1
    mkt_loss = (1 + period_df["Mkt-RF"] + period_df["RF"]).prod() - 1
    print(f"\n---{event}---")
    print(f"Portfolio return:{portfolio_loss:.2%}")
    print(f"Market benchmark: {mkt_loss:.2%}")
    print(f"Underperformance: {portfolio_loss - mkt_loss:.2%}")
"""
5.ASSET-LEVEL VARIANCE ANALISIS
"""
assets = []
for stock in stocks:
  aligned = pd.concat([returns[stock],df],axis=1).dropna()
  new_y = aligned[stock] - aligned["RF"]
  new_reg = LinearRegression().fit(aligned[["Mkt-RF","SMB","HML"]], new_y)
  alpha = new_reg.intercept_*252
  beta = new_reg.coef_
  R_score = new_reg.score(aligned[["Mkt-RF","SMB","HML"]], new_y)
  assets.append({
  f"Ticker": stock,
  "Alpha:":f"{alpha:.4f}",
  "Mkt-RF":f"{beta[0]:.4f}",
  "SMB:":f"{beta[1]:.4f}",
  "HML:":f"{beta[2]:.4f}",
  "R-score":f"{R_score:.4f}"
  })
attribution_df = pd.DataFrame(assets).set_index("Ticker")
print(attribution_df)
"""
6. VISUALIZATION
"""
benchmark_wealth_index = (1+df["Mkt-RF"]+df["RF"]).cumprod()
figure, (ax1,ax2) = plt.subplots(2,1, figsize=(10,8), sharex=True)
ax1.set_title("Portfolio vs Benchmark")
ax1.set_ylabel("Cumulative Growth")
ax1.plot(df.index, wealth_index, label="Portfolio")
ax1.plot(df.index, benchmark_wealth_index, label="Benchmark")
ax1.legend()
ax2.plot(df.index, drawdowns, color="red")
plt.show()




